# Variance ceiling and predictor saturation, within vs outside TED domain

Generalizes the historical `pooled_variance_ceiling_synonymous_calibrated_SE.ipynb`'s
within/outside-TED-domain contrast into the same config-driven form as
[`variance_ceiling_master_file.ipynb`](variance_ceiling_master_file.ipynb): `variant_class`
comes from `config_variant_classes.yaml` and the tool set comes from `config_correlations.yaml`
(via `utils/variant_filtering`), instead of hardcoding filter constants and a predictor list.

Unlike the single-ceiling notebook, this one draws a **paired within/outside-TED-domain
contrast**: for each qualifying gene-trait pair, the same variant-class population is split by
`ted_domain` into two matched sub-populations, and every bootstrap resamples gene-trait pairs
once, applying the *same* resampled indices to both regions -- so the comparison between regions
is paired, not two independent estimates.

## 1. Model and noise definition

For variant $v$ in gene–trait pair $j=(g,t)$,

$$
\hat\beta_{v,j} = X_{v,j} + E_{v,j}.
$$

Define the per-trait noise variance as the empirical variance (ddof=1) of AC=1 synonymous-class
variants on the unassociated (null-trait) panel, pooled by trait only:

$$
\sigma_t^2 = \operatorname{Var}(\hat\beta_{\mathrm{AC=1,syn}}|\mathrm{trait}=t).
$$

Thus $\operatorname{Var}(E_{v,j}) \approx \sigma_t^2$ for any variant $v$ in trait $t$. The main analysis is restricted to
**AC=1** (singleton) variants of the chosen `variant_class`, split into `within_TED` /
`outside_TED` by `ted_domain`, with every predictor in the chosen tool set compared on exactly
the same variants (complete-case) in both regions.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data
CONFIG_DIR   = str(REPO_ROOT / 'configs')
MASTER_PATH  = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
UNASSOC_PATH = env_override('UNASSOC_PATH', fetch_hf_data('genebass_unassociated.parquet', REPO_ROOT))
FIG_DIR      = Path(env_override('FIG_DIR', '../../../paper_figures'))
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import load_config, load_variant_class, scan_variants, pick_annos, env_override

_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

MASTER_SCHEMA = set(pl.scan_parquet(MASTER_PATH).collect_schema().names())
print(f'master table: {len(MASTER_SCHEMA)} cols')

## 2. Parameters

In [ ]:
# --- variant class + tool set (drive everything below; env_override(NAME, default)) ---
variant_class       = env_override('VARIANT_CLASS', 'missense')   # any key in config_variant_classes.yaml
config_file         = env_override('CONFIG_FILE', 'config_correlations.yaml')
selected_categories = env_override('NOISE_CEILING_CATEGORIES', None, 'list')   # None -> this variant class's own `tool_categories`
only_snps           = env_override('ONLY_SNPS', False, bool)

# --- gene-trait pair settings ---
MIN_N_REGION = env_override('MIN_VARIANTS', 10, int)

N_BOOT    = env_override('N_BOOT', 2000, int)
BOOT_SEED = env_override('BOOT_SEED', 1, int)


## 3. Load config, variant class and tool set

In [ ]:
anno_cfg, all_annos = load_config(CONFIG_DIR, config_file)
vc = load_variant_class(CONFIG_DIR, variant_class)
lf = scan_variants(MASTER_PATH, vc, only_snps=only_snps)

if selected_categories is None:
    selected_categories = vc['tool_categories']

annos = pick_annos(anno_cfg, all_annos, selected_categories, MASTER_SCHEMA)
anno_to_label = dict(anno_cfg.select(['annotation', 'label']).unique().iter_rows())

print(f'variant_class = {variant_class}')
print(f'categories    = {selected_categories}')
print(f'{len(annos)} tools: {annos}')

## 4. Per-trait noise from AC=1 synonymous variants on the unassociated panel

The null-trait panel carries no annotation columns (`id`, `region`, `phenotype`, `phenocode`,
`mean_pheno_value`, `SE`, `Pvalue`, `AF`, `n_cases`) and it is **not** restricted to AC=1: only
~43% of its rows are singletons and its `AF` runs all the way up to common. Since
`mean_pheno_value` is the *mean* phenotype over the AC carriers, its variance falls as
$\approx\sigma^2/\mathrm{AC}$, so pooling over all AC would badly under-estimate the singleton
noise. We therefore join `AC` and `consequence_synonymous_variant` in from the master table by
`id` and keep only AC=1 synonymous rows before taking the per-trait variance:

$$\sigma_t^2=\operatorname{Var}\big(\hat\beta_{\mathrm{AC}=1,\,\mathrm{syn}}\mid \mathrm{trait}=t\big).$$


In [ ]:
COL_ID, COL_GENE, COL_TRAIT = 'id', 'gene_name', 'phenotype'
COL_MAC, COL_BETA, COL_SE = 'AC', 'mean_pheno_value', 'SE'
COL_SYNONYMOUS = 'consequence_synonymous_variant'
COL_TED, COL_TED_NA = 'ted_domain', 'ted_domain_is_na'

# The null panel has no AC / consequence columns -- bring them in from the master table by `id`.
annot = (
    pl.scan_parquet(MASTER_PATH)
    .select([COL_ID, COL_MAC, COL_SYNONYMOUS])
    .unique(subset=[COL_ID])
    .collect()
)

unassoc_df = (
    pl.scan_parquet(UNASSOC_PATH)
    .select(pl.col(COL_ID), pl.col(COL_TRAIT), pl.col(COL_BETA).alias('beta'))
    .filter(pl.col('beta').is_not_null(), pl.col('beta').is_finite())
    .collect()
    .join(annot, on=COL_ID, how='inner')
    .filter(pl.col(COL_MAC) == 1, pl.col(COL_SYNONYMOUS) == 1)
)

# Per-trait noise: empirical variance of AC=1 synonymous betas, pooled by trait only.
trait_noise = (
    unassoc_df
    .group_by(COL_TRAIT)
    .agg(pl.col('beta').var(ddof=1).alias('sigma2_t'), pl.len().alias('k_t'))
)

print(f'AC=1 synonymous null rows: {unassoc_df.height:,}')
print(f'Traits: {trait_noise.height}   min variants/trait: {trait_noise["k_t"].min():,}')
s2 = trait_noise['sigma2_t']
print(f'sigma2_t  median={s2.median():.4f}  min={s2.min():.4f}  max={s2.max():.4f}')


### Orthogonal check: are the Genebass SEs honest?

The empirical AC=1 synonymous noise $\sigma_t^2$ and the *reported* $\overline{SE^2}$ for the same
rows are two independent estimates of the same quantity. Their ratio
$c_t=\sigma_t^2/\overline{SE^2}$ is exactly the old calibration factor, now computed per trait. If
$c_t\approx1$ the reported SEs are already well calibrated, and any difference between this
notebook and the SE-based one is *not* attributable to over-confident Genebass SEs.


In [ ]:
se_check = (
    pl.scan_parquet(UNASSOC_PATH)
    .select([COL_ID, COL_TRAIT, pl.col(COL_BETA).alias('beta'), COL_SE])
    .filter(pl.col('beta').is_finite(), pl.col(COL_SE).is_finite())
    .collect()
    .join(annot, on=COL_ID, how='inner')
    .filter(pl.col(COL_MAC) == 1, pl.col(COL_SYNONYMOUS) == 1)
    .group_by(COL_TRAIT)
    .agg(
        pl.col('beta').var(ddof=1).alias('sigma2_t'),
        (pl.col(COL_SE) ** 2).mean().alias('mean_se2'),
    )
    .with_columns((pl.col('sigma2_t') / pl.col('mean_se2')).alias('c_t'))
)

ct = se_check['c_t']
print(f'c_t across {se_check.height} traits: median={ct.median():.4f}  '
      f'q10={ct.quantile(0.1):.4f}  q90={ct.quantile(0.9):.4f}')
print('c_t == 1 would mean the reported Genebass SE is exactly right.')


## 5. Load complete-case singleton variants for the chosen class + tools, split by TED domain

A variant enters the analysis only if it is AC=1, matches `variant_class`, has a known
TED-domain annotation, and every tool in `annos` has a finite score. For each variant we define
$\sigma_v^2 = \sigma_t^2$ (the trait's empirical noise variance), and label it `within_TED`/`outside_TED` from `ted_domain`.


In [ ]:
predictor_exprs = [pl.col(a).cast(pl.Float64) for a in annos]

common_complete_case = (
    lf
    .filter(
        pl.col(COL_MAC) == 1,
        pl.col(COL_BETA).is_not_null(), pl.col(COL_BETA).is_finite(),
        pl.col(COL_SE).is_not_null(), pl.col(COL_SE).is_finite(),
        pl.col(COL_TED_NA) == 0,
        *[pl.col(a).is_not_null() & pl.col(a).is_finite() for a in annos],
    )
    .select(
        pl.col(COL_ID), pl.col(COL_GENE), pl.col(COL_TRAIT),
        pl.col(COL_BETA).cast(pl.Float64).alias('beta'),
        pl.col(COL_SE).cast(pl.Float64).alias('SE'),
        pl.col(COL_TED).cast(pl.Int8),
        *predictor_exprs,
    )
    .collect()
    .join(trait_noise.select([COL_TRAIT, 'sigma2_t']), on=COL_TRAIT, how='left')
    .with_columns(
        pl.col('sigma2_t').alias('sigma2'),
        pl.when(pl.col(COL_TED) == 1).then(pl.lit('within_TED'))
          .otherwise(pl.lit('outside_TED')).alias('region'),
    )
)

print(f'Complete-case {variant_class} singleton rows: {common_complete_case.height:,}')
print(f'Minimum trait noise sigma2: {common_complete_case["sigma2"].min():.6f}')

if common_complete_case.height == 0:
    raise ValueError(f'No complete-case singleton variants for {variant_class!r}')
if common_complete_case['sigma2'].min() <= 0:
    raise ValueError('Trait noise variance is non-positive.')


## 6. Fix the matched gene–trait pairs once

A gene–trait pair is retained only if it has at least `MIN_N_REGION` complete-case variants
**both** within TED and outside TED. This pair set is then fixed for the rest of the notebook.

In [ ]:
counts = (
    common_complete_case
    .group_by(COL_GENE, COL_TRAIT, 'region')
    .agg(pl.len().alias('n_region'))
    .pivot(on='region', index=[COL_GENE, COL_TRAIT], values='n_region')
    .fill_null(0)
)

for region in ('within_TED', 'outside_TED'):
    if region not in counts.columns:
        counts = counts.with_columns(pl.lit(0).alias(region))

common_pairs = (
    counts
    .filter(pl.col('within_TED') >= MIN_N_REGION, pl.col('outside_TED') >= MIN_N_REGION)
    .select(COL_GENE, COL_TRAIT)
    .sort(COL_GENE, COL_TRAIT)
)

analysis = common_complete_case.join(common_pairs, on=[COL_GENE, COL_TRAIT], how='inner')

print(f'Matched gene-trait pairs: {common_pairs.height:,}')
print(f'Analysis rows:            {analysis.height:,}')

## 7. Common detectable variance using the per-trait noise

For a gene–trait pair $j$ with $n_j$ variants (computed independently in each region),

$$
SS_{\mathrm{total},j}=\sum_v(\hat\beta_{v,j}-\bar\beta_j)^2,
\qquad
SS_{\mathrm{noise},j}=\left(1-\frac{1}{n_j}\right)\sum_v \sigma_v^2,
\qquad
SS_{\mathrm{detectable},j}=SS_{\mathrm{total},j}-SS_{\mathrm{noise},j}.
$$

Pooled across pairs with weight $w_j=n_j-1$, this is the **variance ceiling** for that region.

In [ ]:
def build_ceiling_strata(df: pl.DataFrame, region: str) -> pl.DataFrame:
    rows = []
    d_region = df.filter(pl.col('region') == region)
    for key, d in d_region.partition_by([COL_GENE, COL_TRAIT], as_dict=True).items():
        gene, trait = key
        y = d['beta'].to_numpy().astype(float)
        sigma2 = d['sigma2'].to_numpy().astype(float)
        n = len(y)
        yc = y - y.mean()

        SS_total = float(np.dot(yc, yc))
        SS_noise = float((1.0 - 1.0 / n) * sigma2.sum())

        rows.append({
            COL_GENE: gene, COL_TRAIT: trait, 'n': n, 'w': float(n - 1),
            'SS_total': SS_total, 'SS_noise': SS_noise, 'SS_detect': SS_total - SS_noise,
        })
    return pl.DataFrame(rows).sort(COL_GENE, COL_TRAIT)


def pooled_ceiling_summary(strata: pl.DataFrame) -> dict:
    W = float(strata['w'].sum())
    V_total = float(strata['SS_total'].sum()) / W
    V_noise = float(strata['SS_noise'].sum()) / W
    V_detect = float(strata['SS_detect'].sum()) / W
    return {
        'n_pairs': strata.height, 'W': W,
        'V_total': V_total, 'V_noise': V_noise, 'V_detect': V_detect,
        'R2_max': V_detect / V_total if V_total > 0 else np.nan,
    }


ceiling_within = build_ceiling_strata(analysis, 'within_TED')
ceiling_outside = build_ceiling_strata(analysis, 'outside_TED')

assert ceiling_within.height == ceiling_outside.height
assert (
    ceiling_within.select(COL_GENE, COL_TRAIT).to_dicts()
    == ceiling_outside.select(COL_GENE, COL_TRAIT).to_dicts()
)

ceiling_point = pl.DataFrame([
    {'region': 'within_TED', **pooled_ceiling_summary(ceiling_within)},
    {'region': 'outside_TED', **pooled_ceiling_summary(ceiling_outside)},
])
ceiling_point

## 8. Bootstrap the common ceiling across matched gene–trait pairs

The gene–trait pair is the resampling unit, and within each bootstrap replicate the same sampled pair indices are used for within TED and outside TED.

In [ ]:
def paired_ceiling_bootstrap(within: pl.DataFrame, outside: pl.DataFrame,
                              n_boot: int = 2000, seed: int = 1) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    J = within.height

    def one(d, idx):
        w = d['w'].to_numpy()[idx]
        SS_total = d['SS_total'].to_numpy()[idx]
        SS_noise = d['SS_noise'].to_numpy()[idx]
        SS_detect = d['SS_detect'].to_numpy()[idx]
        W = w.sum()
        return {
            'V_total': SS_total.sum() / W, 'V_noise': SS_noise.sum() / W,
            'V_detect': SS_detect.sum() / W,
            'R2_max': (SS_detect.sum() / W) / (SS_total.sum() / W) if SS_total.sum() > 0 else np.nan,
        }

    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, J, size=J)
        a = one(within, idx)
        o = one(outside, idx)
        rows.append({
            'bootstrap': b,
            'V_detect_within': a['V_detect'], 'R2_max_within': a['R2_max'],
            'V_detect_outside': o['V_detect'], 'R2_max_outside': o['R2_max'],
        })
    return pl.DataFrame(rows)


def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return np.quantile(x, [alpha / 2, 0.5, 1 - alpha / 2])


ceiling_boot = paired_ceiling_bootstrap(ceiling_within, ceiling_outside, n_boot=N_BOOT, seed=BOOT_SEED)

for col in ['V_detect_within', 'V_detect_outside', 'R2_max_within', 'R2_max_outside']:
    print(col, bootstrap_ci(ceiling_boot[col]))

In [ ]:
ceiling_plot = pl.DataFrame([
    {
        'region': 'within TED',
        'V_detect': ceiling_point.filter(pl.col('region') == 'within_TED')['V_detect'].item(),
        'lo': bootstrap_ci(ceiling_boot['V_detect_within'])[0],
        'hi': bootstrap_ci(ceiling_boot['V_detect_within'])[2],
    },
    {
        'region': 'outside TED',
        'V_detect': ceiling_point.filter(pl.col('region') == 'outside_TED')['V_detect'].item(),
        'lo': bootstrap_ci(ceiling_boot['V_detect_outside'])[0],
        'hi': bootstrap_ci(ceiling_boot['V_detect_outside'])[2],
    },
])

ceiling_plot_pd = ceiling_plot.to_pandas()
ceiling_plot_pd['region'] = pd.Categorical(
    ceiling_plot_pd['region'], categories=ceiling_plot_pd['region'].tolist(), ordered=True,
)

(
    ggplot(ceiling_plot_pd, aes(x='region', y='V_detect'))
    + geom_pointrange(aes(ymin='lo', ymax='hi'))
    + geom_hline(yintercept=0, linetype='dashed')
    + labs(x='', y='Pooled detectable variance', title=f'Common detectable signal ({vc.get("x_label", variant_class)})')
    + _THEME
    + theme(figure_size=(5, 5))
)

## 9. Predictor decomposition with the per-trait noise

For predictor score $S_v$, fit separately within each gene–trait pair (independently per
region): $\hat\beta_v = \hat b + \hat a S_v + r_v$. After centering the score,
$x_v=S_v-\bar S$, the fitted-score projection has leverage $h_v = x_v^2/\sum_u x_u^2$. Under
independent heteroskedastic noise, the noise the fit spuriously captures is
$SS_{\mathrm{captured,noise}}=\sum_v h_v\sigma_v^2$, so

$$
SS_{\mathrm{captured}} = SS_{\mathrm{captured,raw}} - SS_{\mathrm{captured,noise}},
\qquad
SS_{\mathrm{uncaptured}} = SS_{\mathrm{resid}} - SS_{\mathrm{resid,noise}},
$$

with the exact identity $SS_{\mathrm{detectable}} = SS_{\mathrm{captured}} + SS_{\mathrm{uncaptured}}$.

In [ ]:
def fit_predictor_by_pair(df: pl.DataFrame, score_col: str) -> pl.DataFrame:
    rows = []
    for key, d in df.partition_by([COL_GENE, COL_TRAIT], as_dict=True).items():
        gene, trait = key
        y = d['beta'].to_numpy().astype(float)
        s = d[score_col].to_numpy().astype(float)
        sigma2 = d['sigma2'].to_numpy().astype(float)
        n = len(y)

        yc = y - y.mean()
        x_v = s - s.mean()

        SS_total = float(np.dot(yc, yc))
        SS_noise = float((1.0 - 1.0 / n) * sigma2.sum())
        sum_x2 = float(np.dot(x_v, x_v))

        if sum_x2 <= 0:
            slope = 0.0
            fitted_c = np.zeros_like(yc)
            SS_cap_noise = 0.0
        else:
            slope = float(np.dot(x_v, yc) / sum_x2)
            fitted_c = slope * x_v
            h_v = x_v**2 / sum_x2
            SS_cap_noise = float(np.sum(h_v * sigma2))

        resid = yc - fitted_c
        SS_reg = float(np.dot(fitted_c, fitted_c))
        SS_resid = float(np.dot(resid, resid))

        SS_resid_noise = SS_noise - SS_cap_noise
        SS_detect = SS_total - SS_noise
        SS_captured = SS_reg - SS_cap_noise
        SS_uncaptured = SS_resid - SS_resid_noise

        rows.append({
            COL_GENE: gene, COL_TRAIT: trait, 'n': n, 'w': float(n - 1), 'slope': slope,
            'SS_total': SS_total, 'SS_reg': SS_reg, 'SS_resid': SS_resid,
            'SS_noise': SS_noise, 'SS_cap_noise': SS_cap_noise,
            'SS_resid_noise': SS_resid_noise, 'SS_detect': SS_detect,
            'SS_captured': SS_captured, 'SS_uncaptured': SS_uncaptured,
        })
    return pl.DataFrame(rows).sort(COL_GENE, COL_TRAIT)

### Algebraic sanity check

For every gene-trait pair: $SST=SSR+SSE$ and $SS_{\mathrm{detectable}}=SS_{\mathrm{captured}}+SS_{\mathrm{uncaptured}}$.

In [ ]:
example_fit = fit_predictor_by_pair(
    analysis.filter(pl.col('region') == 'within_TED'), annos[0],
)

raw_error = example_fit.select(
    (pl.col('SS_total') - pl.col('SS_reg') - pl.col('SS_resid')).abs().max()
).item()
corrected_error = example_fit.select(
    (pl.col('SS_detect') - pl.col('SS_captured') - pl.col('SS_uncaptured')).abs().max()
).item()

print('Max raw OLS identity error:', raw_error)
print('Max corrected identity error:', corrected_error)

## 10. Pool predictor performance

For a region, with $W=\sum_j(n_j-1)$:
$V_{\mathrm{captured}}=\sum_j SS_{\mathrm{captured},j}/W$,
$V_{\mathrm{uncaptured}}=\sum_j SS_{\mathrm{uncaptured},j}/W$,
$V_{\mathrm{detectable}}=\sum_j SS_{\mathrm{detectable},j}/W$.
The saturation fraction is $F_{\mathrm{captured}}=V_{\mathrm{captured}}/V_{\mathrm{detectable}}$.
Because the analysis variants are identical for all predictors, $V_{\mathrm{detectable}}$ is the
same denominator for every predictor within a region.

In [ ]:
def pooled_predictor_summary(strata: pl.DataFrame) -> dict:
    W = float(strata['w'].sum())
    V_total = float(strata['SS_total'].sum()) / W
    V_noise = float(strata['SS_noise'].sum()) / W
    V_detect = float(strata['SS_detect'].sum()) / W
    V_captured = float(strata['SS_captured'].sum()) / W
    V_uncaptured = float(strata['SS_uncaptured'].sum()) / W
    return {
        'V_total': V_total, 'V_noise': V_noise, 'V_detect': V_detect,
        'V_captured': V_captured, 'V_uncaptured': V_uncaptured,
        'R2_max': V_detect / V_total if V_total > 0 else np.nan,
        'F_captured': V_captured / V_detect if V_detect != 0 else np.nan,
    }

## 11. Paired gene–trait bootstrap for every predictor

The bootstrap resamples matched gene–trait pairs with replacement; for a given predictor the same sampled pair indices are applied to within TED and outside TED. The corrected variance components are never clipped.

In [ ]:
def paired_predictor_bootstrap(within: pl.DataFrame, outside: pl.DataFrame,
                                n_boot: int = 2000, seed: int = 1) -> pl.DataFrame:
    assert within.height == outside.height
    assert (
        within.select(COL_GENE, COL_TRAIT).to_dicts()
        == outside.select(COL_GENE, COL_TRAIT).to_dicts()
    )

    rng = np.random.default_rng(seed)
    J = within.height

    def one(d, idx):
        w = d['w'].to_numpy()[idx]
        SS_detect = d['SS_detect'].to_numpy()[idx]
        SS_captured = d['SS_captured'].to_numpy()[idx]
        SS_uncaptured = d['SS_uncaptured'].to_numpy()[idx]
        W = w.sum()
        V_detect = SS_detect.sum() / W
        V_captured = SS_captured.sum() / W
        V_uncaptured = SS_uncaptured.sum() / W
        return {
            'V_detect': V_detect, 'V_captured': V_captured, 'V_uncaptured': V_uncaptured,
            'F_captured': V_captured / V_detect if V_detect != 0 else np.nan,
        }

    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, J, size=J)
        a = one(within, idx)
        o = one(outside, idx)
        rows.append({
            'bootstrap': b,
            'F_within': a['F_captured'], 'F_outside': o['F_captured'],
            'V_captured_within': a['V_captured'], 'V_captured_outside': o['V_captured'],
            'V_uncaptured_within': a['V_uncaptured'], 'V_uncaptured_outside': o['V_uncaptured'],
        })
    return pl.DataFrame(rows)


within_data = analysis.filter(pl.col('region') == 'within_TED')
outside_data = analysis.filter(pl.col('region') == 'outside_TED')

all_results = []

for i, a in enumerate(annos):
    label = anno_to_label.get(a, a)

    within_fit = fit_predictor_by_pair(within_data, a)
    outside_fit = fit_predictor_by_pair(outside_data, a)
    assert within_fit.height == common_pairs.height
    assert outside_fit.height == common_pairs.height

    within_point = pooled_predictor_summary(within_fit)
    outside_point = pooled_predictor_summary(outside_fit)

    assert np.isclose(
        within_point['V_detect'],
        ceiling_point.filter(pl.col('region') == 'within_TED')['V_detect'].item(),
        rtol=1e-10, atol=1e-12,
    )
    assert np.isclose(
        outside_point['V_detect'],
        ceiling_point.filter(pl.col('region') == 'outside_TED')['V_detect'].item(),
        rtol=1e-10, atol=1e-12,
    )

    boot = paired_predictor_bootstrap(within_fit, outside_fit, n_boot=N_BOOT, seed=BOOT_SEED + i)

    F_in_ci = bootstrap_ci(boot['F_within'])
    F_out_ci = bootstrap_ci(boot['F_outside'])
    Vcap_in_ci = bootstrap_ci(boot['V_captured_within'])
    Vcap_out_ci = bootstrap_ci(boot['V_captured_outside'])

    all_results.append({
        'predictor': label, 'n_pairs': common_pairs.height,
        'F_within': within_point['F_captured'], 'F_within_lo': F_in_ci[0], 'F_within_hi': F_in_ci[2],
        'F_outside': outside_point['F_captured'], 'F_outside_lo': F_out_ci[0], 'F_outside_hi': F_out_ci[2],
        'V_captured_within': within_point['V_captured'],
        'V_captured_within_lo': Vcap_in_ci[0], 'V_captured_within_hi': Vcap_in_ci[2],
        'V_captured_outside': outside_point['V_captured'],
        'V_captured_outside_lo': Vcap_out_ci[0], 'V_captured_outside_hi': Vcap_out_ci[2],
        'V_uncaptured_within': within_point['V_uncaptured'],
        'V_uncaptured_outside': outside_point['V_uncaptured'],
        'V_detect_within': within_point['V_detect'], 'V_detect_outside': outside_point['V_detect'],
        'R2_max_within': within_point['R2_max'], 'R2_max_outside': outside_point['R2_max'],
    })

all_results = pl.DataFrame(all_results)
all_results

## 12. Sanity check: the ceiling is identical across predictors

The `Vdetect_*`/`R2max_*` columns below should be constant across all rows.

In [ ]:
all_results.select('predictor', 'V_detect_within', 'V_detect_outside', 'R2_max_within', 'R2_max_outside')

In [ ]:
# Saturation fraction with CI, both regions, sorted by within-TED
with pl.Config(tbl_rows=25, fmt_str_lengths=40, tbl_width_chars=200):
    display(
        all_results.select(
            'predictor',
            pl.col('F_within').round(4), pl.col('F_within_lo').round(4), pl.col('F_within_hi').round(4),
            pl.col('F_outside').round(4), pl.col('F_outside_lo').round(4), pl.col('F_outside_hi').round(4),
        ).sort('F_within', descending=True)
    )


## 13. Main figure — pooled detectable and captured variance per predictor

Two bar plots (within TED, outside TED), each ordered independently by that region's own
$V_{\mathrm{captured}}$ (plotnine's `facet_wrap` shares one categorical order across panels, so
an independent per-region order means two separate plots rather than facets). The first bar in
each plot is $V_{\mathrm{detectable}}$ (the common denominator from Section 7); the remaining
bars are each predictor's $V_{\mathrm{captured}}$, in the same units, each with a gene-trait-pair
bootstrap CI on that quantity directly (not on the $F_{\mathrm{captured}}$ ratio). A right-hand
axis shows the same values as a percentage of that region's $V_{\mathrm{detectable}}$.

In [ ]:
v_detect_within = ceiling_point.filter(pl.col('region') == 'within_TED')['V_detect'].item()
v_detect_outside = ceiling_point.filter(pl.col('region') == 'outside_TED')['V_detect'].item()
v_detect_within_ci = bootstrap_ci(ceiling_boot['V_detect_within'])
v_detect_outside_ci = bootstrap_ci(ceiling_boot['V_detect_outside'])


def build_region_df(v_detect, v_detect_ci, vcap_col, vcap_lo_col, vcap_hi_col):
    '''One region's bar data, ordered by that region's own V_captured (descending).'''
    d = all_results.sort(vcap_col, descending=True)
    rows = [{'label': 'Detectable', 'V': v_detect, 'lo': v_detect_ci[0], 'hi': v_detect_ci[2], 'kind': 'Detectable'}]
    for pred, v, lo, hi in zip(
        d['predictor'].to_list(), d[vcap_col].to_list(), d[vcap_lo_col].to_list(), d[vcap_hi_col].to_list(),
    ):
        rows.append({'label': pred, 'V': v, 'lo': lo, 'hi': hi, 'kind': 'Captured'})
    df = pd.DataFrame(rows)
    df['label'] = pd.Categorical(df['label'], categories=df['label'].tolist(), ordered=True)
    return df


within_df = build_region_df(
    v_detect_within, v_detect_within_ci, 'V_captured_within', 'V_captured_within_lo', 'V_captured_within_hi',
)
outside_df = build_region_df(
    v_detect_outside, v_detect_outside_ci, 'V_captured_outside', 'V_captured_outside_lo', 'V_captured_outside_hi',
)


def build_bar_plot(df, title):
    return (
        ggplot(df, aes(x='label', y='V', fill='kind'))
        + geom_col()
        + geom_errorbar(aes(ymin='lo', ymax='hi'), width=0.3)
        + geom_hline(yintercept=0, linetype='dotted')
        # + scale_fill_manual(values={'Detectable': '#AFDC2E', 'Captured': '#2A78D6'})
        + scale_fill_manual(values={'Detectable': '#9F72BB', 'Captured': '#3DAED4'})
        + labs(x='', y='Variance ceiling', title=title)
        + _THEME
        + theme(
            figure_size=(0.25*len(df) + 1, 4),
            axis_text_x=element_text(size=12, rotation=45, ha='right'),
            axis_text_y=element_text(size=12),
            axis_title=element_text(size=12),
            legend_position='none',
        )
    )


def show_with_pct_axis(p, v_detect):
    '''plotnine has no secondary-axis scale, so this is added directly on the matplotlib
    figure plotnine draws.'''
    fig = p.draw()
    ax = fig.axes[0]
    sec = ax.secondary_yaxis(
        'right', color='gray',
        functions=(lambda v: v / v_detect * 100, lambda pct: pct / 100 * v_detect),
    )
    sec.set_ylabel('% of detectable variance')
    return fig


p_within = show_with_pct_axis(build_bar_plot(within_df, f'within TED ({variant_class})'), v_detect_within)
p_within.savefig(FIG_DIR / f'F4_variance_ceiling_{variant_class}_within_TED.svg', dpi=200, bbox_inches='tight')
p_within

In [ ]:
v_detect_within_ci

In [ ]:
v_detect_outside_ci

In [ ]:
p_outside = show_with_pct_axis(build_bar_plot(outside_df, f'outside TED ({variant_class})'), v_detect_outside)
p_outside.savefig(FIG_DIR / f'F4_variance_ceiling_{variant_class}_outside_TED.svg', dpi=200, bbox_inches='tight')
p_outside

In [ ]:
within_df